# Robust BO入門：実行時の入力ずれとリスクを考慮する

設定した入力 `X` が実験装置や製造工程で正確に再現されるとは限りません。このNotebookでは、**設定値（nominal design）から実行時に入力がずれる場合**を扱います。

入力摂動 → GP予測 → リスク集約 → 制約された探索範囲で候補提案 → 実測値の追加 → 再学習 → 次の候補提案、という流れを学びます。

**重要：** 実行時の入力ずれに対する頑健性は、訓練データの外れ値への頑健性とは別です。また、摂動シナリオ数 `n_w` は候補数 `q` や入力次元 `d` とは異なります。


## Robust BOで何を頑健にするか

- **Nominal design**：最適化器が決める設定値 `x`
- **Perturbation**：実行時に加わるずれ `δ`
- **Scenario**：実際に評価される入力 `x+δ`
- **Risk measure**：複数のシナリオの品質を1つの尺度にまとめる規則

通常のBOが `f(x)` の改善を探すのに対し、この例では `f(x+δ)` の**シナリオにわたる期待値など**を重視します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.acquisition.risk_measures import Expectation as BoTorchExpectation
from botorch.models.transforms.input import InputPerturbation
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.objectives import CVaR, Expectation, MeanVariance, WorstCase
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Nominal design problem and execution perturbations

候補は1次元nominal designです。実行時には`-0.04, 0, +0.04`のずれが起きるとします。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return 1.0 - 6.0 * (X - 0.62).square() + 0.12 * torch.sin(8 * torch.pi * X)


perturbations = torch.tensor([[-0.04], [0.0], [0.04]])
n_w = perturbations.shape[0]
bounds = torch.tensor([[0.08], [0.92]])
train_X = torch.linspace(0.10, 0.90, 11).unsqueeze(-1)
train_Y = response(train_X)

print("nominal input dimension:", train_X.shape[-1])
print("number of perturbation scenarios:", n_w)

### ずれを考慮した目的関数を可視化する

同じnominal入力でも、`-0.04`、`0`、`+0.04` の3つの実行条件で応答が異なります。以下では元の応答と、3シナリオの平均・最悪値を比較します。

探索範囲を `[0.08,0.92]` としているため、各摂動後の入力も `[0,1]` の内部にあります。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(bounds[0, 0], bounds[1, 0], 201).unsqueeze(-1)
scenario_grid = plot_X.unsqueeze(-2) + perturbations.unsqueeze(0)
scenario_responses = response(scenario_grid).squeeze(-1)
nominal_responses = response(plot_X).squeeze(-1)
mean_responses = scenario_responses.mean(dim=-1)
worst_responses = scenario_responses.min(dim=-1).values
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), nominal_responses, label="Nominal response")
ax.plot(plot_X.squeeze(-1), mean_responses, label="Scenario mean")
ax.plot(plot_X.squeeze(-1), worst_responses, label="Scenario worst case")
ax.set(xlabel="Nominal x", ylabel="Response", title="Effect of execution perturbations")
ax.legend()
plt.show()

## 3. InputPerturbation expands candidates before posterior evaluation

physical perturbationはraw/public design spaceで適用します。`n_w`を入力次元へ追加するのではありません。

In [ ]:
input_transform = InputPerturbation(perturbation_set=perturbations)
model = SingleTaskGP(
    train_X,
    train_Y,
    input_transform=input_transform,
)
model.fit()
model.eval()

nominal_X = torch.tensor([[0.55]])
expanded_X = input_transform(nominal_X)
print("nominal shape:", nominal_X.shape)
print("expanded scenario shape:", expanded_X.shape)
print("expanded scenarios:\n", expanded_X)

### InputPerturbationが展開する軸

`InputPerturbation` は1つのnominal候補を3つのシナリオへ展開します。入力次元が3次元になるわけではありません。

`SingleTaskGP` は既存の `fit()` で学習します。入力変換はモデルに渡し、Notebook内で独自のGP学習処理を実装しません。

## 4. Posterior sampling keeps MC and scenario axes distinct

`InputPerturbation`によりposteriorのq-like row axisへscenarioが展開されます。risk objectiveが`n_w`個を1つのnominal candidate utilityへ戻します。

In [ ]:
sampler = SobolQMCNormalSampler(torch.Size([64]), seed=101)
posterior = model.posterior(nominal_X)
samples = sampler(posterior)

print("posterior mean shape:", posterior.mean.shape)
print("MC sample shape:", samples.shape)
assert samples.shape[0] == 64
assert samples.shape[-2] == n_w

### MCサンプルとシナリオは異なる

`SobolQMCNormalSampler` はGP事後分布からサンプルを生成します。先頭の64は**MCサンプル数**であり、摂動シナリオ数ではありません。

シナリオ軸は候補ごとの `n_w=3` 行です。リスク指標はこのシナリオ軸を集約し、nominal候補1点の効用へ戻します。

## 5. Scenario-risk semantics

robotorchanのrisk measuresはscenario valuesの最終軸を集約する汎用APIです。BoTorchの`MCAcquisitionObjective`とは別の責務です。

In [ ]:
scenario_values = response(expanded_X).squeeze(-1)
risk_values = {
    "expectation": Expectation()(scenario_values),
    "mean_variance": MeanVariance(risk_weight=2.0)(scenario_values),
    "worst_case": WorstCase()(scenario_values),
    "cvar": CVaR(alpha=0.8)(scenario_values),
}

print("scenario values:", scenario_values)
for name, value in risk_values.items():
    print(name, value.item())

### Expectation・MeanVariance・WorstCase・CVaRの違い

- `Expectation`：シナリオの平均的な応答を重視
- `MeanVariance`：平均とばらつきのトレードオフを評価
- `WorstCase`：最も悪いシナリオを重視
- `CVaR`：悪い側のシナリオ群に注目するリスク尺度

どの尺度が適切かは、平均性能を重視するか、不良・失敗を避けたいかによって変わります。**ここでのリスク指標の数値と獲得関数値は異なります**。

## 6. Robust acquisition uses a scenario-aware BoTorch objective

E2E candidate optimizationではBoTorch-native `Expectation(n_w=...)`をMC acquisitionへ渡します。これによりscenario axisを平均してnominal candidate utilityに戻します。

### リスクを考慮した獲得関数

`qLogNoisyExpectedImprovement` にBoTorchの `Expectation(n_w=...)` を渡すと、**GP事後サンプルのシナリオ軸**を平均してから改善を評価できます。

上の `robotorchan.objectives.Expectation` は観測済みのシナリオ値を集約する汎用APIで、BoTorchの `MCAcquisitionObjective` とは役割が異なります。

最適化器が返すのは3シナリオではなく、**nominal候補1点**です。

In [ ]:
robust_objective = BoTorchExpectation(n_w=n_w)
acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=202),
    objective=robust_objective,
    prune_baseline=False,
)
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, 1])
print("robust nominal candidate:", candidate)
print("acquisition value:", value)

## 7. Candidate evaluation remains scenario-aware

optimizerが返すのは`n_w`行ではなくnominal candidate 1点です。実行時riskを確認するときだけperturbation scenariosへ展開します。

In [ ]:
candidate_scenarios = input_transform(candidate)
candidate_values = response(candidate_scenarios).squeeze(-1)
expected_value = Expectation()(candidate_values)
worst_value = WorstCase()(candidate_values)

assert candidate_scenarios.shape == torch.Size([n_w, 1])
print("nominal candidate:", candidate.squeeze(-1))
print("scenario responses:", candidate_values)
print("scenario expectation:", expected_value.item())
print("scenario worst case:", worst_value.item())

### 提案した候補の実行時リスクを確認する

`candidate` をシナリオへ展開し、合成目的関数で3通りの応答を計算します。期待値と最悪値を確認できますが、**これらをそのままGPの訓練ターゲットとして追加するわけではありません**。

## 8. Update data with what was actually observed

robust acquisitionのrisk utilityをtraining targetへ自動的に書き換えません。実験・シミュレーションで何を観測するかはdata-generating processの契約です。ここではnominal responseを1観測として追加します。

In [ ]:
observed_Y = response(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, observed_Y], dim=0)

assert updated_X.shape[0] == train_X.shape[0] + 1
print("observed nominal response:", observed_Y.item())

### 観測データを追加した後の再学習

この合成例では、nominal設定で測定した `response(candidate)` を新しい観測値として追加します。実務で実際の入力ずれが測定できる場合は、どの座標を入力として記録するかをデータ生成過程に合わせて設計してください。

観測を追加した後、同じ `InputPerturbation` を用いるGPを再学習し、既存の獲得関数から次のnominal候補を提案します。

In [ ]:
updated_transform = InputPerturbation(perturbation_set=perturbations)
updated_model = SingleTaskGP(updated_X, updated_Y, input_transform=updated_transform)
updated_model.fit()
updated_model.eval()
updated_acquisition = qLogNoisyExpectedImprovement(
    model=updated_model,
    X_baseline=updated_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=203),
    objective=BoTorchExpectation(n_w=n_w),
    prune_baseline=False,
)
next_candidate, next_value = optimize_acqf(
    acq_function=updated_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)
assert next_candidate.shape == (1, 1)
assert torch.isfinite(next_value).all()
print("Next nominal candidate:", next_candidate.detach())
print("Next acquisition value:", next_value.detach())

### 最初と次の候補を応答曲線上で確認する

nominal候補を図示します。**応答曲線は合成例の真値であり、実際のBOでは未知**です。提案位置を比較するための参考として表示します。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), nominal_responses, label="Nominal response")
ax.plot(plot_X.squeeze(-1), mean_responses, label="Scenario expectation")
ax.plot(plot_X.squeeze(-1), worst_responses, label="Scenario worst case")
ax.axvline(candidate.detach().item(), linestyle="--", label="First robust candidate")
ax.axvline(next_candidate.detach().item(), linestyle=":", label="Next robust candidate")
ax.set(xlabel="Nominal x", ylabel="Response", title="Robust BO candidate proposals")
ax.legend()
plt.show()

## 9. Initialization remains nominal

optimizer initial conditionsは`[num_restarts, q, d]`です。scenario数`n_w`を`q`や`d`へ追加しません。

```text
IC: [restart, q, d]
          ↓
InputPerturbation
          ↓
posterior scenario rows: q * n_w
```

candidate constraintsも原則としてnominal design coordinatesに対して定義します。

## 10. Protected structural dimensions

Mixed / MultiTask / MultiFidelityなどでは、categorical、task、fidelity、context、hierarchy等のstructural coordinatesを物理perturbationの対象にしません。continuous design/environmental coordinatesだけを揺らし、protected dimensionsのunionを保持します。

## 11. Robustness layers are independent

| Problem | Layer |
| --- | --- |
| training outlier / heavy tail | robust observation surrogate |
| uncertain measured X | uncertain-input surrogate |
| execution-time X deviation | input perturbation scenarios |
| expectation / CVaR / worst case | decision risk aggregation |
| noisy Y | observation-noise / noisy acquisition semantics |

robust surrogateを使うだけでCVaR optimizationにはならず、risk objectiveを使うだけでtraining outlierへ頑健になるわけでもありません。

## 12. TuRBO and high-dimensional boundaries

TuRBOのtrust regionはnominal public input spaceにあります。scenario axisはTuRBO dimensionや`batch_size`へ追加しません。PCA/PLS等でも物理perturbationはraw design spaceへ適用してからmodel-owned reductionへ進めます。latent offsetを物理的不確かさとして暗黙に扱いません。

## 13. Related documentation

- `docs-internal/contracts/input_perturbation_contract.md`
- `docs/optimization/objective-posterior-transform.md`
- `docs/theory/14_robust_gaussian_process.md`
- `docs/theory/16_uncertain_input_gp.md`
- `36_objective_posterior_transform.ipynb`
- `47_turbo_trust_region.ipynb`

Robust BOでは「何が不確かか」と「どの軸をどこで集約するか」を明示することが最重要です。

## 14. 実務に向けたAPI整理

| 目的 | 既存API | 注意点 |
|---|---|---|
| GP学習 | `SingleTaskGP.fit()` | 観測済みデータから学習 |
| 摂動の定義 | `InputPerturbation` | nominal入力をscenarioへ展開 |
| 事後サンプル | `SobolQMCNormalSampler` | MC軸とscenario軸は別 |
| リスク集約 | `robotorchan.objectives.Expectation` など | 観測済みscenario値の集約 |
| MC獲得関数のリスク目的 | `botorch.acquisition.risk_measures.Expectation` | `n_w` シナリオを集約 |
| 改善の評価 | `qLogNoisyExpectedImprovement` | 観測値やリスク値と混同しない |
| nominal候補の提案 | `optimize_acqf(q=1)` | 戻り値は1点のnominal入力 |

**重要：** Robust BOは、どの不確かさを扱うか、どの軸でシナリオを集約するか、実際に何を観測してGPへ登録するかを分けて設計します。